# BankScope whole-table GPU retrieval evaluation

This Colab notebook reproduces the missing full-corpus experiment on a T4 GPU:

1. upload and extract `bankscope_colab_gpu_bundle.zip`;
2. install the minimal runtime dependencies;
3. validate the frozen 5,565-record corpus and 30-query evaluation contract;
4. generate all Qwen3 document embeddings;
5. evaluate BM25, dense, and equal-weight RRF hybrid retrieval;
6. package the embeddings, result JSON, and environment provenance for download.

Use **Runtime → Change runtime type → T4 GPU** before running all cells. No OpenAI or Hugging Face token is required because the Qwen model is public.

## 1. Parameters

Batch size 8 is conservative for a Colab T4 with the 2,048-token BankScope limit. Increase it only after a successful run.

In [ ]:
from pathlib import Path

BUNDLE_FILENAME = "bankscope_colab_gpu_bundle.zip"
EXTRACT_ROOT = Path("/content/bankscope_gpu_project")
BATCH_SIZE = 8
MAX_SEQUENCE_LENGTH = 2048
CANDIDATE_K = 30
RRF_K = 60

print({
    "batch_size": BATCH_SIZE,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "candidate_k": CANDIDATE_K,
    "rrf_k": RRF_K,
})

## 2. Upload and extract the project bundle

Select the supplied ZIP when the upload dialog opens. Re-running the cell safely overwrites files in the ephemeral Colab working directory.

In [ ]:
import shutil
from google.colab import files

uploaded = files.upload()
if BUNDLE_FILENAME in uploaded:
    bundle_path = Path("/content") / BUNDLE_FILENAME
else:
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError(f"Upload exactly one ZIP bundle; received: {list(uploaded)}")
    bundle_path = Path("/content") / zip_names[0]

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(bundle_path, EXTRACT_ROOT)
PROJECT_ROOT = EXTRACT_ROOT

required_paths = [
    PROJECT_ROOT / "scripts/embed.py",
    PROJECT_ROOT / "scripts/evaluate.py",
    PROJECT_ROOT / "src/bankscope",
    PROJECT_ROOT / "data/processed/chunks.jsonl",
    PROJECT_ROOT / "data/processed/tables.jsonl",
    PROJECT_ROOT / "data/processed/manifest.json",
    PROJECT_ROOT / "data/evaluation/queries.jsonl",
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f"Bundle is incomplete: {missing}")

print(f"Project extracted to: {PROJECT_ROOT}")

## 3. Install the frozen evaluation runtime

The repository package itself targets Python 3.13, while Colab may use another supported runtime. We therefore install only the required libraries and run the bundled source through `PYTHONPATH`; this does not change the retrieval implementation.

In [ ]:
import subprocess
import sys

requirements = [
    "numpy>=2.0,<3.0",
    "pandas>=2.0,<4.0",
    "beautifulsoup4==4.15.0",
    "lxml==6.1.1",
    "sec2md==0.1.23",
    "bm25s==0.2.14",
    "sentence-transformers==5.6.1",
    "transformers==5.14.1",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *requirements],
    check=True,
)
print("Dependencies installed.")

## 4. Verify the T4 GPU and environment

In [ ]:
import importlib.metadata as metadata
import platform
import torch

subprocess.run(["nvidia-smi"], check=True)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a T4 GPU runtime and reconnect.")

gpu_name = torch.cuda.get_device_name(0)
environment = {
    "python": platform.python_version(),
    "gpu": gpu_name,
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "packages": {name: metadata.version(name) for name in [
        "numpy", "bm25s", "sentence-transformers", "transformers", "sec2md"
    ]},
}
print(environment)
if "T4" not in gpu_name.upper():
    print(f"Note: expected a T4, but Colab assigned {gpu_name}; the run can still continue.")

## 5. Validate the frozen corpus and qrels

This check fails before the expensive embedding run if the uploaded corpus is incomplete or does not match its recorded manifest.

In [ ]:
import hashlib
import json

chunks_path = PROJECT_ROOT / "data/processed/chunks.jsonl"
tables_path = PROJECT_ROOT / "data/processed/tables.jsonl"
manifest_path = PROJECT_ROOT / "data/processed/manifest.json"
queries_path = PROJECT_ROOT / "data/evaluation/queries.jsonl"
embeddings_path = PROJECT_ROOT / "data/processed/embeddings.npz"
results_path = PROJECT_ROOT / "data/evaluation/results/retrieval.json"

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as input_file:
        for block in iter(lambda: input_file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
chunk_count = sum(1 for line in chunks_path.open(encoding="utf-8") if line.strip())
table_count = sum(1 for line in tables_path.open(encoding="utf-8") if line.strip())
queries = [json.loads(line) for line in queries_path.open(encoding="utf-8") if line.strip()]

assert chunk_count == manifest["chunk_count"] == 5565
assert table_count == manifest["table_count"] == 1783
assert sha256(chunks_path) == manifest["outputs"]["chunks"]["sha256"]
assert sha256(tables_path) == manifest["outputs"]["tables"]["sha256"]
assert len(queries) == 30
assert sum(query["status"] == "answerable" for query in queries) == 28

print({
    "retrieval_records": chunk_count,
    "complete_tables": table_count,
    "queries": len(queries),
    "answerable_queries": sum(query["status"] == "answerable" for query in queries),
    "chunks_sha256": sha256(chunks_path),
})

## 6. Generate all Qwen3 embeddings on the GPU

This is the long-running cell. It validates every input length, encodes all 5,565 records in corpus order, normalizes the 1,024-dimensional vectors, and writes provenance plus the source SHA-256 into `embeddings.npz`.

In [ ]:
import os

run_environment = os.environ.copy()
run_environment["PYTHONPATH"] = str(PROJECT_ROOT / "src")
run_environment["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
run_environment["TOKENIZERS_PARALLELISM"] = "true"

embed_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts/embed.py"),
    "--input", str(chunks_path),
    "--output", str(embeddings_path),
    "--batch-size", str(BATCH_SIZE),
    "--max-seq-length", str(MAX_SEQUENCE_LENGTH),
    "--overwrite",
]
subprocess.run(embed_command, cwd=PROJECT_ROOT, env=run_environment, check=True)
print(f"Embedding archive: {embeddings_path} ({embeddings_path.stat().st_size / 1024**2:.1f} MiB)")

## 7. Evaluate BM25, dense, and RRF hybrid retrieval

The evaluator verifies record order, model revision, embedding shape, and the exact corpus hash before searching.

In [ ]:
evaluate_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts/evaluate.py"),
    "--qrels", str(queries_path),
    "--chunks", str(chunks_path),
    "--tables", str(tables_path),
    "--embeddings", str(embeddings_path),
    "--output", str(results_path),
    "--methods", "dense", "bm25", "hybrid",
    "--candidate-k", str(CANDIDATE_K),
    "--rrf-k", str(RRF_K),
]
subprocess.run(evaluate_command, cwd=PROJECT_ROOT, env=run_environment, check=True)

## 8. Review the measured result

The historical sec2md-v3 hybrid row is shown only as a comparison point. The values for the whole-table corpus are read from the newly generated result file.

In [ ]:
import pandas as pd

evaluation = json.loads(results_path.read_text(encoding="utf-8"))
rows = []
for method, metrics in evaluation["summary"].items():
    query_count = int(metrics["query_count"])
    rows.append({
        "method": method,
        "Hit@1": round(metrics["hit_rate_at_1"] * query_count),
        "Hit@3": round(metrics["hit_rate_at_3"] * query_count),
        "Hit@5": round(metrics["hit_rate_at_5"] * query_count),
        "Hit@10": round(metrics["hit_rate_at_10"] * query_count),
        "MRR@10": metrics["mrr_at_10"],
        "complete_cross_bank@10": metrics.get("complete_group_hit_rate_at_10"),
    })

rows.append({
    "method": "historical sec2md-v3 hybrid",
    "Hit@1": 12, "Hit@3": 20, "Hit@5": 23, "Hit@10": 24,
    "MRR@10": 0.5889880952380953,
    "complete_cross_bank@10": 1 / 3,
})
summary_frame = pd.DataFrame(rows).set_index("method")
display(summary_frame.style.format({"MRR@10": "{:.3f}", "complete_cross_bank@10": "{:.3f}"}))

hybrid_rows = [row for row in evaluation["per_query"] if row["method"] == "hybrid"]
hybrid_misses = [
    {
        "query_id": row["query_id"],
        "question_type": row["question_type"],
        "first_relevant_rank": row["metrics"]["first_relevant_rank"],
        "group_recall_at_10": row["metrics"].get("group_recall_at_10"),
    }
    for row in hybrid_rows
    if not row["metrics"]["hit_at_5"] or row["metrics"].get("complete_group_hit_at_10") == 0
]
display(pd.DataFrame(hybrid_misses))

## 9. Package and download the reproducibility artifacts

The output ZIP contains the complete embedding archive, retrieval result, corpus manifest, qrels, and GPU/package provenance. Keep it outside Git unless a later decision explicitly selects smaller tracked result files.

In [ ]:
from datetime import datetime, timezone

output_directory = Path("/content/bankscope_gpu_outputs")
output_directory.mkdir(parents=True, exist_ok=True)

for source in [embeddings_path, results_path, manifest_path, queries_path]:
    shutil.copy2(source, output_directory / source.name)

run_provenance = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "environment": environment,
    "parameters": {
        "batch_size": BATCH_SIZE,
        "max_sequence_length": MAX_SEQUENCE_LENGTH,
        "candidate_k": CANDIDATE_K,
        "rrf_k": RRF_K,
    },
    "sha256": {
        "chunks.jsonl": sha256(chunks_path),
        "tables.jsonl": sha256(tables_path),
        "queries.jsonl": sha256(queries_path),
        "embeddings.npz": sha256(embeddings_path),
        "retrieval.json": sha256(results_path),
    },
}
(output_directory / "run_provenance.json").write_text(
    json.dumps(run_provenance, indent=2) + "\n", encoding="utf-8"
)

archive_base = Path("/content/bankscope_gpu_results")
archive_path = Path(shutil.make_archive(str(archive_base), "zip", output_directory))
print(f"Created: {archive_path} ({archive_path.stat().st_size / 1024**2:.1f} MiB)")
files.download(str(archive_path))

## Next decision

Use the measured per-method and per-query output to decide whether equal-weight hybrid is sufficient or whether the next bounded experiment should be weighted RRF and/or conservative rerank fusion. Do not tune on the result before preserving this baseline artifact.